# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ailya-Shah/INTERNSHIP-TASKS/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup -- connect to the warehouse (same mid-panel month as w03)

Reusing March 2026 (`month = '2026-03'`) so this notebook's leakage findings are directly comparable to w03's. Still never touching the sealed `_sample` (June 2026).

In [1]:
%pip install -q duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT    = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY90 = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
MONTH   = "2026-03"
print("Connected. Using month:", MONTH)


Connected. Using month: 2026-03


## 1. Build the feature vector

Full engineered feature vector for the Ranking Signal Analysis lane: content properties, keyword/demand signals, categoricals (one-hot), and query-mix signals from `fact_content_query_90d` -- tested for window alignment rather than assumed safe (see Section 3). `content_age_days` is clipped at 0 to fix a real bug w03 surfaced: some `dim_content` rows have `content_created_date` after the March window (dim_content is a full snapshot, not month-sliced), which produced impossible negative ages.

In [3]:
# 1a. Label + exposure aggregate for March 2026 (same construction as w03)
page_agg = con.sql(f"""
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id)                              AS client_hash_id,
        SUM(gsc_impressions)                                   AS impressions_win,
        SUM(gsc_clicks)                                        AS clicks_win,
        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0)                  AS avg_position_win
    FROM {FACT}
    WHERE month = '{MONTH}' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()

page_agg["is_page_one"] = page_agg["avg_position_win"].between(1, 10).astype(int)
page_agg = page_agg[page_agg["impressions_win"] >= 100]
print(f"labeled pages: {len(page_agg):,} | base rate: {page_agg['is_page_one'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

labeled pages: 101,441 | base rate: 56.5%


In [4]:
# 1b. Content + keyword features -- age/freshness derived, clipped at 0 (fixes the negative-age bug from w03)
content = con.sql(f"""
    SELECT
        content_hash_id,
        word_count, char_count,
        search_volume, competition, competition_level, cpc,
        content_type, main_intent,
        backlinks, category_count,
        keyword_char_count, keyword_token_count, url_char_count,
        GREATEST(date_diff('day', content_created_date, DATE '{MONTH}-01'), 0) AS content_age_days,
        GREATEST(date_diff('day', content_updated_date, DATE '{MONTH}-01'), 0) AS days_since_last_update
    FROM {CONTENT}
    WHERE is_published = TRUE AND is_deleted = FALSE
""").df()

n_clipped = (con.sql(f"""
    SELECT COUNT(*) FROM {CONTENT}
    WHERE date_diff('day', content_created_date, DATE '{MONTH}-01') < 0
""").fetchone()[0])
print(f"rows with content_created_date AFTER the window start (clipped to age=0): {n_clipped:,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows with content_created_date AFTER the window start (clipped to age=0): 113,174


In [5]:
# 1c. Query-mix features -- context columns repeat per row, so ANY_VALUE them, never SUM/COUNT(DISTINCT)
qmix = con.sql(f"""
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count)     AS visible_queries,
        ANY_VALUE(rare_impressions_share)          AS rare_share,
        ANY_VALUE(anonymized_impressions_share)    AS anon_share,
        MAX(impressions_90d) * 1.0
            / NULLIF(SUM(impressions_90d), 0)      AS top_query_share
    FROM {QUERY90}
    GROUP BY content_hash_id
""").df()

data = page_agg.merge(content, on="content_hash_id", how="left").merge(qmix, on="content_hash_id", how="left")
print(f"full feature frame: {data.shape[0]:,} rows x {data.shape[1]} cols")
data.head(3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

full feature frame: 101,441 rows x 25 cols


,content_hash_id,client_hash_id,impressions_win,clicks_win,avg_position_win,is_page_one,word_count,char_count,search_volume,competition,...,category_count,keyword_char_count,keyword_token_count,url_char_count,content_age_days,days_since_last_update,visible_queries,rare_share,anon_share,top_query_share
0,content_22c063002b7c1caf,client_73cda7b4e4f265ea,314.0,1.0,10.057325,0,<NA>,<NA>,40,0.07,...,0.0,40.0,8.0,130.0,366.0,0.0,3.0,0.165706,0.743516,0.523810
1,content_6cf70a5cb30e76f2,client_73cda7b4e4f265ea,3465.0,5.0,7.392496,1,2893,19316,40,0.00,...,0.0,26.0,4.0,118.0,366.0,0.0,11.0,0.014551,0.882451,0.390638
2,content_0d510f7a6abb761e,client_73cda7b4e4f265ea,1431.0,5.0,5.828092,1,2516,16813,110,0.03,...,0.0,25.0,4.0,95.0,366.0,0.0,18.0,0.024042,0.638524,0.490472


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUM_COLS = ["word_count", "char_count", "content_age_days", "days_since_last_update",
            "search_volume", "competition", "cpc", "backlinks", "category_count",
            "keyword_char_count", "keyword_token_count", "url_char_count",
            "visible_queries", "top_query_share", "rare_share", "anon_share"]
CAT_COLS = ["content_type", "main_intent", "competition_level"]

# Missingness follows content_type (per flyrank-data skill) -- flag before imputing, never blind fillna(0)
for col in ["search_volume", "competition", "cpc", "backlinks",
            "keyword_char_count", "keyword_token_count",
            "visible_queries", "top_query_share", "rare_share", "anon_share"]:
    data[f"{col}_is_missing"] = data[col].isna().astype(int)

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), NUM_COLS),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT_COLS),
])

print("Numeric features:", len(NUM_COLS), "| Categorical features:", len(CAT_COLS))
print("Missing-flag columns added:", [c for c in data.columns if c.endswith("_is_missing")])


Numeric features: 16 | Categorical features: 3
Missing-flag columns added: ['search_volume_is_missing', 'competition_is_missing', 'cpc_is_missing', 'backlinks_is_missing', 'keyword_char_count_is_missing', 'keyword_token_count_is_missing', 'visible_queries_is_missing', 'top_query_share_is_missing', 'rare_share_is_missing', 'anon_share_is_missing']


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Categorical? | Available before decision moment? |
|---|---|---|---|---|
| `word_count`, `char_count` | content length | median impute + `_is_missing` flag | no | yes -- static content property |
| `content_age_days` | days since publish, clipped at 0 | none needed (derived) | no | yes -- but see bug note below |
| `days_since_last_update` | days since last edit, clipped at 0 | none needed (derived) | no | yes |
| `search_volume`, `competition`, `cpc` | external keyword demand | median impute + `_is_missing` flag (missingness follows `content_type`, confirmed in w03/flyrank-data) | no | yes -- independent demand-side data |
| `content_type`, `main_intent`, `competition_level` | categorical metadata | most-frequent impute, one-hot | **yes** | yes -- static metadata |
| `backlinks`, `category_count` | off-page / structural signals | median impute + flag | no | yes |
| `keyword_char_count`, `keyword_token_count`, `url_char_count` | keyword/URL structure | median impute + flag | no | yes |
| `visible_queries`, `top_query_share`, `rare_share`, `anon_share` | query-mix (from `fact_content_query_90d`) | median impute + flag | no | **borderline -- tested explicitly in Section 3** |

**Bug caught and fixed:** a nontrivial number of `dim_content` rows have `content_created_date` *after* my March 2026 window start -- because `dim_content` is a full snapshot across the whole warehouse history, not sliced to my month. Left unclipped this produces impossible negative ages. I clip at 0 (`GREATEST(..., 0)`), which is an honest fix but not a perfect one -- a page with `content_age_days = 0` could mean "published today" or "published after my window and this number is meaningless." I'm flagging this rather than hiding it; a future pass would filter these rows out entirely instead of clipping.

**Why query-mix is "borderline," not simply "safe":** `fact_content_query_90d` is a fixed trailing 90-day window. Depending on exactly which 90 days it covers relative to March 2026, it could partially overlap my label's own outcome month -- which would make it a soft version of the same leakage risk as `gsc_clicks`. Rather than assume it away, Section 3 tests it directly: with vs. without, same way I tested `clicks_win` in w03.

In [7]:
# Supporting check: does search_volume/competition missingness really follow content_type?
missingness_by_type = data.groupby("content_type")[["search_volume", "backlinks"]].apply(lambda g: g.isna().mean())
missingness_by_type


,search_volume,backlinks
content_type,,
comparison article,0.000000,0.001003
feedly article,1.000000,1.000000
keyword article,0.007839,0.364147


## 3. The leakage hunt

Three deliberate attacks on my own feature set, each isolating one column/group and comparing AUC with vs. without it on the same grouped-by-client split. A big jump toward ~1.0 is a confession; a small jump is a healthy sign the feature is safe.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def evaluate(feature_cols, cat_cols, label, groups):
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                           ("scale", StandardScaler())]), [c for c in feature_cols if c not in cat_cols]),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])

    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    train_idx, test_idx = next(gss.split(data[feature_cols], label, groups))

    pipe.fit(data.iloc[train_idx][feature_cols], label.iloc[train_idx])
    proba = pipe.predict_proba(data.iloc[test_idx][feature_cols])[:, 1]
    return roc_auc_score(label.iloc[test_idx], proba)

y = data["is_page_one"]
groups = data["client_hash_id"]

BASE_NUM = ["word_count", "char_count", "content_age_days", "days_since_last_update",
            "search_volume", "competition", "cpc", "backlinks", "category_count"]
BASE_CAT = ["content_type", "main_intent", "competition_level"]
QMIX_COLS = ["visible_queries", "top_query_share", "rare_share", "anon_share"]

honest_auc = evaluate(BASE_NUM + BASE_CAT, BASE_CAT, y, groups)
print(f"Attack 0 -- honest baseline (no borderline/leaky columns):        AUC = {honest_auc:.3f}")


Attack 0 -- honest baseline (no borderline/leaky columns):        AUC = 0.627


In [9]:
# Attack 1 -- the label itself, avg_position_win, added as a feature (textbook leakage: should collapse toward ~1.0)
auc_with_label = evaluate(BASE_NUM + BASE_CAT + ["avg_position_win"], BASE_CAT, y, groups)
print(f"Attack 1 -- WITH the label itself (avg_position_win) as a feature: AUC = {auc_with_label:.3f}   <- textbook confession")
print(f"            gap vs honest: {auc_with_label - honest_auc:+.3f}")


Attack 1 -- WITH the label itself (avg_position_win) as a feature: AUC = 0.991   <- textbook confession
            gap vs honest: +0.364


In [10]:
# Attack 2 -- gsc_clicks, downstream of position (same trap as w03, reproduced here on the full feature set)
auc_with_clicks = evaluate(BASE_NUM + BASE_CAT + ["clicks_win"], BASE_CAT, y, groups)
print(f"Attack 2 -- WITH clicks_win (downstream of position):              AUC = {auc_with_clicks:.3f}")
print(f"            gap vs honest: {auc_with_clicks - honest_auc:+.3f}")


Attack 2 -- WITH clicks_win (downstream of position):              AUC = 0.671
            gap vs honest: +0.044


In [11]:
# Attack 3 -- the borderline query-mix block, tested honestly rather than assumed safe
auc_with_qmix = evaluate(BASE_NUM + BASE_CAT + QMIX_COLS, BASE_CAT, y, groups)
print(f"Attack 3 -- WITH query-mix features (visible_queries, etc.):      AUC = {auc_with_qmix:.3f}")
print(f"            gap vs honest: {auc_with_qmix - honest_auc:+.3f}")

print("\nVerdict:")
print(f"  Attack 1 (label itself)  gap = {auc_with_label - honest_auc:+.3f}  -> {'CONFIRMED leak' if auc_with_label - honest_auc > 0.15 else 'smaller than expected -- investigate'}")
print(f"  Attack 2 (clicks_win)    gap = {auc_with_clicks - honest_auc:+.3f}  -> {'meaningful leak, exclude' if auc_with_clicks - honest_auc > 0.03 else 'small -- borderline, exclude to be safe'}")
print(f"  Attack 3 (query-mix)     gap = {auc_with_qmix - honest_auc:+.3f}  -> {'meaningful shift, treat as borderline/exclude' if abs(auc_with_qmix - honest_auc) > 0.03 else 'small -- reasonably safe to KEEP as features'}")


Attack 3 -- WITH query-mix features (visible_queries, etc.):      AUC = 0.798
            gap vs honest: +0.171

Verdict:
  Attack 1 (label itself)  gap = +0.364  -> CONFIRMED leak
  Attack 2 (clicks_win)    gap = +0.044  -> meaningful leak, exclude
  Attack 3 (query-mix)     gap = +0.171  -> meaningful shift, treat as borderline/exclude


**Reading the three attacks:** Attack 1 is the confession test -- feeding the model the label itself should produce the largest, most obvious jump, and it's my proof that this evaluation harness actually *detects* leakage rather than just always reporting a similar number. Attacks 2 and 3 are the real judgment calls: `clicks_win` is excluded regardless of the exact gap size, because it's *structurally* downstream of position (confirmed leak by construction, not just by the number). The query-mix decision follows whatever the printed verdict says -- if the gap is small, I keep those four features in my honest model; if it's large, I treat query-mix the same as `clicks_win` and exclude it until I can verify the 90-day window's exact date alignment against my March label window.

## 4. What I excluded and why

| Excluded field | Why |
|---|---|
| `gsc_avg_position`, `gsc_sum_position` (`avg_position_win`) | **Is the label.** Confirmed by Attack 1 -- including it produces the textbook leakage jump. |
| `gsc_clicks` (`clicks_win`) | Structurally downstream of search position; confirmed leaky in both w03 and Attack 2 here. |
| `provider_used`, `model_used` | Dictionary explicitly bans these as model features (process/product metadata, not a content signal). |
| `content_hash_id`, `client_hash_id` | Grouping/joining/splitting keys only -- pseudonymous IDs carry no real signal and must never be features. |
| `content_created_date`, `content_updated_date` (raw dates) | Only the *derived* ages (`content_age_days`, `days_since_last_update`) are kept as features; the raw calendar dates themselves are dropped once derived, since a raw date is closer to an identifier than a signal. |
| `last_optimized_date`, `optimization_eligible_date` | Smell like product-decision fields (someone/something decided to optimize this page) -- excluded per the "product flags are outputs, never inputs" rule, even though they weren't explicitly tested here. |
| query-mix features (`visible_queries`, `top_query_share`, `rare_share`, `anon_share`) | **Conditionally excluded** -- kept only if Attack 3's gap came back small; otherwise excluded until the 90-day window's date alignment against my label window is verified precisely. |

In [12]:
# Final honest feature list, decided by the leakage hunt above (not by intuition)
FINAL_NUM_FEATURES = BASE_NUM
FINAL_CAT_FEATURES = BASE_CAT
if abs(auc_with_qmix - honest_auc) <= 0.03:
    FINAL_NUM_FEATURES = FINAL_NUM_FEATURES + QMIX_COLS
    print("Query-mix features KEPT -- gap was small enough to trust.")
else:
    print("Query-mix features EXCLUDED -- gap too large to trust without further window-alignment verification.")

print("\nFinal honest feature set:")
print("  numeric/derived:", FINAL_NUM_FEATURES)
print("  categorical:    ", FINAL_CAT_FEATURES)


Query-mix features EXCLUDED -- gap too large to trust without further window-alignment verification.

Final honest feature set:
  numeric/derived: ['word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'search_volume', 'competition', 'cpc', 'backlinks', 'category_count']
  categorical:     ['content_type', 'main_intent', 'competition_level']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.